---
## 1. The NIJ Recidivism Forecasting Challenge

The [National Institute of Justice's Recidivism Forecasting Challenge](https://nij.ojp.gov/funding/recidivism-forecasting-challenge) asked teams to predict whether a person released on parole in Georgia would be arrested for a new felony or misdemeanor crime within **3 years** of their supervision start date.

We will work with the same data. The files are already on the server:

| Path | Contents |
|------|----------|
| `/home/mlfp_shared/nij_data/train_preprocessed.csv` | Training set — preprocessed features + outcome |
| `/home/mlfp_shared/nij_data/test_no_outcomes.csv` | Test set — preprocessed features, no outcome, use this to make your predictions |

### Codebook

| # | Variable | Description |
|---|----------|-------------|
| 1 | `ID` | Unique person ID |
| 2 | `Gender` | M / F |
| 3 | `Race` | Black or White |
| 4 | `Age_at_Release` | Age group at prison release (18-22, 23-27, … 48+) |
| 5 | `Residence_PUMA` | Census PUMA of residence at release |
| 6 | `Gang_Affiliated` | Verified gang affiliation |
| 7 | `Supervision_Risk_Score_First` | First parole risk score (1–10, 1=lowest) |
| 8 | `Supervision_Level_First` | First supervision level (Standard / High / Specialized) |
| 9 | `Education_Level` | Education at prison entry |
| 10 | `Dependents` | Number of dependents at prison entry |
| 11 | `Prison_Offense` | Primary conviction offense group |
| 12 | `Prison_Years` | Years in prison before release |
| 13–20 | `Prior_Arrest_Episodes_*` | Prior arrest counts by charge type |
| 21–28 | `Prior_Conviction_Episodes_*` | Prior conviction counts by charge type |
| 29–30 | `Prior_Revocations_*` | Prior parole/probation revocations |
| 31–33 | `Condition_*` | Parole release conditions |
| 34–37 | `Violations_*` | Supervision violations |
| 38–49 | *(see below)* | Delinquency reports, program attendances, drug tests, employment |
| 50 | `Recidivism_Within_3years` | **Outcome**: new arrest within 3 years of supervision start |

**Variables 34–49** (violations, drug tests, employment) were measured *during* the parole supervision episode — before the 3-year recidivism window. However, consider: do any of these variables feel like they might overlap with or be caused by the outcome you are trying to predict? Visit the [challenge page](https://nij.ojp.gov/funding/recidivism-forecasting-challenge) to read how the NIJ defined the measurement windows and think carefully about whether any variables could be leaking information about future outcomes.


---
## 2. Load the Data

We'll start by loading the **raw** training data so we can explore the original variable values. The raw data is available at:

```
/home/mlfp_shared/nij_data/nij-challenge2021_training_dataset.csv
```


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_TRAIN_PATH = Path('/home/mlfp_shared/nij_data/nij-challenge2021_training_dataset.csv')
OUTCOME        = 'Recidivism_Within_3years'

train_raw = pd.read_csv(RAW_TRAIN_PATH)

print('Shape:', train_raw.shape)
print('\nOutcome distribution:')
print(train_raw[OUTCOME].value_counts())


Take a moment to look at the data. Notice that many variables are stored as strings even though they represent counts or ordered categories — for example, `Prior_Arrest_Episodes_Felony` takes values like `'0'`, `'1'`, `'10 or more'`. Before fitting any model, these would need to be converted to numeric values and any remaining categoricals one-hot encoded. We'll handle that in Section 4, but first let's explore the raw data.

Check the column types and null counts:


In [ ]:
print(train_raw.dtypes)
print('\nNull counts:')
print(train_raw.isnull().sum()[train_raw.isnull().sum() > 0])


---
## 3. Exploring Features

Before fitting any model, it is useful to understand how individual features relate to the outcome. The function below groups the data by the two outcome values (`Yes` / `No`) and shows the mean of a given feature in each group. For numeric features, a difference in means suggests the feature is informative. For string features it shows the count per group so you can spot imbalances.

We run this on the **raw** data so the original variable names from the codebook still work.


In [ ]:
def crosstab_with_outcome(df, feature, outcome=OUTCOME):
    """
    For numeric features: returns the mean of `feature` per outcome group (Yes / No).
    For string features: returns n and the proportion of 'Yes' outcomes per category,
    sorted by that proportion descending.
    """
    if pd.api.types.is_numeric_dtype(df[feature]):
        return (
            df.groupby(outcome)[feature]
            .mean()
            .round(3)
            .rename(f'mean({feature})')
            .to_frame()
        )
    else:
        counts = df.groupby(feature)[outcome].value_counts().unstack(fill_value=0)
        counts['n'] = counts.sum(axis=1)
        counts['prop_yes'] = (counts.get('Yes', 0) / counts['n']).round(3)
        return counts[['n', 'prop_yes']].sort_values('prop_yes', ascending=False)


Try it on a few features. Change the feature name to explore others:


In [ ]:
crosstab_with_outcome(train_raw, '<pick a feature>')

---
## 4. Load the Preprocessed Data

For the raw data, fitting a linear model would require encoding all those string columns — converting counts like `'10 or more'` to integers, one-hot encoding categoricals like `Age_at_Release`, imputing missing values, and aligning the train and test column schemas. In practice this is an important step you would do yourself, but for this tutorial we have done it for you.

> **What the preprocessing does:**
> - `Yes`/`No` columns → `1`/`0`
> - Count string columns (e.g. `'3 or more'`) → numeric, using the lower bound of the capped bin
> - All remaining string columns (age group, education, offense type, etc.) → one-hot encoded
> - Missing values → median (numeric) or mode (categorical)
> - Train and test columns aligned so they share the same schema

The script that does this is `src/preprocess.py` in the repo if you want to inspect it.

Load the preprocessed files now:


In [ ]:
TRAIN_PATH = Path('/home/mlfp_shared/nij_data/train_preprocessed.csv')
TEST_PATH  = Path('/home/mlfp_shared/nij_data/test_no_outcomes.csv')

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print('Preprocessed train shape:', train.shape)
print('Preprocessed test shape: ', test.shape)
print('\nSample columns:', list(train.columns[:10]))


---
## 5. Train a Logistic Regression Model

### 5.1 Split Training Data

How should we split train vs. test? Does a simple split here work or do we need to worry about time leakage?

Since we don't have time information, we will use a simple split, but note that that could be an issue. Hold out 20 % of the training data for validation. We will use this later to compare models.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in train.columns if c != OUTCOME]
X = train[feature_cols].values
y = train[OUTCOME].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:     ', X_train.shape[0])
print('Validation size:', X_val.shape[0])


### 5.2 Choose Your Features

Pick **about 5 features** from the codebook that you think will be most predictive. Replace the list below with your choices. The model will raise an error if you select more than 10.


In [ ]:
feature_cols

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

# --- FILL IN YOUR FEATURE NAMES HERE (max 10) ---
selected_features = [
    'Gang_Affiliated',
 'Supervision_Risk_Score_First',
 'Dependents',
 'Prior_Arrest_Episodes_Felony',
 'Prior_Arrest_Episodes_Misd',
 'Prior_Arrest_Episodes_Violent',
 'Prior_Arrest_Episodes_Property',
 'Prior_Arrest_Episodes_Drug',
 '_v1',
 'Prior_Arrest_Episodes_DVCharges',
 'Prior_Arrest_Episodes_GunCharges',
 'Prior_Conviction_Episodes_Felony',
 'Prior_Conviction_Episodes_Misd',
 'Prior_Conviction_Episodes_Viol',
 'Prior_Conviction_Episodes_Prop',
 'Prior_Conviction_Episodes_Drug',
 '_v2',
 '_v3',
 '_v4',
 'Prior_Revocations_Parole',
 'Prior_Revocations_Probation',
 'Condition_MH_SA',
 'Condition_Cog_Ed',
 'Condition_Other',
 'Violations_ElectronicMonitoring',
 'Violations_Instruction',
 'Violations_FailToReport',
 'Violations_MoveWithoutPermission',
 'Delinquency_Reports',
 'Program_Attendances',
 'Program_UnexcusedAbsences',
 'Residence_Changes',
 'Avg_Days_per_DrugTest',
 'DrugTests_THC_Positive',
 'DrugTests_Cocaine_Positive',
 'DrugTests_Meth_Positive',
 'DrugTests_Other_Positive',
 'Percent_Days_Employed',
 'Jobs_Per_Year',
 'Employment_Exempt'
    # '<feature1>',
    # '<feature2>',
]
# ------------------------------------------------

# assert len(selected_features) <= 10, (
#     f'You selected {len(selected_features)} features — the maximum is 10. '
#     'Remove some features and re-run.'
# )
missing = [f for f in selected_features if f not in feature_cols]
assert not missing, f'Features not found in data: {missing}'

idx = [feature_cols.index(f) for f in selected_features]

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train[:, idx], y_train)

val_prob_lr = lr.predict_proba(X_val[:, idx])[:, 1]
print(f'LR validation AUC: {roc_auc_score(y_val, val_prob_lr):.4f}')
print('AUC does not depend on a threshold — but all other metrics do.')
print('Use plot_threshold_metrics() below to choose a threshold before reporting them.')


### 5.3 Choosing a Classification Threshold

**Important:** `sklearn`'s `.predict()` method always uses a threshold of 0.5 — it has no way of knowing whether 0.5 is a sensible cutoff for your problem. You should never use `.predict()` directly without first deciding what threshold is appropriate and applying it yourself.

A logistic regression outputs a probability. Turning that into a 0/1 prediction requires choosing a threshold $t$: predict 1 if $p \geq t$, else 0. Different thresholds trade off precision against recall:

- **Lower $t$** → flag more people as high-risk → higher recall, lower precision
- **Higher $t$** → flag fewer people → higher precision, lower recall

The right threshold depends on the cost of each type of error, which is a policy decision, not a statistical one. The function below plots accuracy, precision, recall, and F1 across 101 thresholds so you can see the full picture before committing to a value.


In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

def plot_threshold_metrics(y_true, y_prob):
    """
    Plot accuracy, precision, recall, and F1 over 101 thresholds (0.00 to 1.00).
    Use the plot to pick a threshold that reflects your priorities.
    """
    thresholds = np.linspace(0, 1, 101)
    metrics = {'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []}

    for t in thresholds:
        preds = (np.array(y_prob) >= t).astype(int)
        metrics['Accuracy'].append(accuracy_score(y_true, preds))
        metrics['Precision'].append(precision_score(y_true, preds, zero_division=np.nan))
        metrics['Recall'].append(recall_score(y_true, preds, zero_division=0))
        metrics['F1'].append(f1_score(y_true, preds, zero_division=0))

    fig, ax = plt.subplots(figsize=(8, 4))
    for name, values in metrics.items():
        ax.plot(thresholds, values, label=name)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Classification metrics vs. decision threshold')
    ax.legend(loc='lower left')
    plt.tight_layout()
    plt.show()


Call `plot_threshold_metrics` on your validation set predictions. Look at the plot before deciding what threshold to use when reporting performance:


In [ ]:
plot_threshold_metrics(y_val, val_prob_lr)

When you increase your threshold (move from left to right), does that mean you are predicting more or fewer people will recidivate within 3 years?

How does your models performance change as you increase your threshold?

As a sanity check, it's often helpful to look at the extremes of the plot (thresholds of 0 and 1):

In [ ]:
threshold = 0
print(f'Threshold = {threshold}')
preds = (np.array(val_prob_lr) >= threshold).astype(int)
print(f'\tRecall: {recall_score(y_val, preds, zero_division=0):.2f}')
print(f'\tPrecision: {precision_score(y_val, preds, zero_division=np.nan):.2f}')
print(f'\tBaserate of outcomes (% of val set = 1): {y_val.mean():.2f}')

print("-"*100)
threshold = 0.98
print(f'Threshold = {threshold}')
preds = (np.array(val_prob_lr) >= threshold).astype(int)
print(f'\tRecall: {recall_score(y_val, preds, zero_division=0):.2f}')
print(f'\tPrecision when threshold is {threshold}: {precision_score(y_val, preds, zero_division=np.nan):.2f}')

print("-"*100)
threshold = 1
print(f'Threshold = {threshold}')
preds = (np.array(val_prob_lr) >= threshold).astype(int)
print(f'\tRecall: {recall_score(y_val, preds, zero_division=0):.2f}')
print(f'\tPrecision when threshold is {threshold}: {precision_score(y_val, preds, zero_division=np.nan):.2f}')

What do you notice about these numbers? 

For example, why is the precision at threshold of 0 the same as the baserate (hint: think about [how precision is calculated](https://en.wikipedia.org/wiki/Precision_and_recall))?

---
## 6. Fitting a More Complex Model

Logistic regression with 5 features is a good starting point, but we might want to:

- Use **all** features
- Tune the regularization strength (lasso penalty)
- Try more powerful models like random forests or gradient-boosted trees

These models can take significantly longer to fit. In this section we will tune a lasso logistic regression across a penalty grid. The function is deliberately slowed down to simulate a real long-running job — this gives you a reason to practice keeping a process alive after disconnecting.


### 6.1 Running Longer Jobs

We'll simulate a longer-running job to illustrate this point. We're going to tune a lasso logistic regression with different levels of regularization.

You must provide a list of `C` values — the regularization parameter used directly by scikit-learn's `LogisticRegression`. To keep server load low, limit your grid to at most 10 values.

What are some reasonable values to choose for `C`? What does a smaller `C` value do in practice? If you're unsure, check the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

First, run the command below **without** any session manager. Once it starts, disconnect from the server (close the VS Code Remote-SSH window or type `exit` in a terminal). Then reconnect and check whether the process finished. Pass your `C` grid as space-separated values on the command line. **[server]**


In [ ]:
!python src/run_slow_fit.py 0.0001 0.001 0.01 0.1 1.0


After reconnecting, check whether the process is still running:


In [ ]:
# [server] — run this in a terminal after reconnecting
!ps aux | grep run_slow_fit


The process was killed when your session ended. This is the core problem `screen` solves.

### 6.2 `screen`: Keep Processes Alive Across Disconnects

`screen` is a terminal multiplexer that keeps a session running on the server independently of your SSH connection. The key advantage for our purposes: you can start a long Python job inside a `screen` session, detach from it, disconnect from the server entirely, and the job keeps running.

Start a named screen session **[server]**:


In [ ]:
screen -S lasso_fit


You are now inside the screen session. Activate your venv and start the slow fit **[server]**:


In [ ]:
source .venv/bin/activate
python src/run_slow_fit.py # your C grid specified as positional arguments; for example: 0.01 0.05


While the model is fitting, **detach** from the screen session — this leaves it running:

```
Ctrl-a  then  d
```

You are back at the normal server prompt. Now fully disconnect **[server]**:


In [ ]:
exit


Reconnect and verify the job is still running **[local → server]**:


In [ ]:
ssh mlfp
screen -ls                  # list all screen sessions
screen -r lasso_fit         # reattach to your session


While waiting for the fit to finish, here are the essential `screen` commands:

| Key / Command | Action |
|---------------|--------|
| `Ctrl-a d` | Detach from session (leaves it running) |
| `screen -ls` | List all running sessions |
| `screen -r <name>` | Reattach to a session |
| `screen -S <name>` | Start a new named session |
| `Ctrl-a c` | Create a new window inside the session |
| `Ctrl-a n` | Switch to the next window |
| `Ctrl-a p` | Switch to the previous window |
| `Ctrl-a k` | Kill the current window |
| `exit` (inside session) | Exit screen entirely |

Practice detaching and reattaching a few times while the model runs.

Once the job finishes, reattach to collect the results:


In [ ]:
screen -r lasso_fit
# The results will be printed to the terminal
# When you are done, exit the session:
exit


> **Note:** `src/run_slow_fit.py` fits a lasso grid and writes results to `results_lasso.json` in your project directory. Pass your `C` values as command-line arguments — run `python src/run_slow_fit.py --help` to see usage.


---
## 7. Model Selection

You now have two candidate models:

1. **Logistic regression (selected features)** — fitted in Section 5
2. **Lasso logistic regression (all features, tuned)** — fitted in Section 6

Use `plot_threshold_metrics()` on each model's validation probabilities to understand the full threshold-metric trade-off before choosing a threshold and comparing them. Remember: **do not use `.predict()` to evaluate your models** — apply your chosen threshold manually so the comparison is fair and intentional.


In [ ]:
# Load the lasso grid results saved by run_slow_fit.py
import json as _json
results_lasso = _json.loads(Path('results_lasso.json').read_text())

# Refit the best lasso model on the training split so we can get val probabilities
best_C = max(results_lasso, key=lambda r: r['val_auc'])['C']
best_lasso = LogisticRegression(
    solver='saga', l1_ratio=1, C=best_C,
    max_iter=5000, random_state=42
)
best_lasso.fit(X_train, y_train)
val_prob_lasso = best_lasso.predict_proba(X_val)[:, 1]

print('=== LR (selected features) ===')
plot_threshold_metrics(y_val, val_prob_lr)

print('=== Lasso (all features) ===')
plot_threshold_metrics(y_val, val_prob_lasso)

print('\nLasso grid results:')
print(pd.DataFrame(results_lasso).to_string(index=False))


After inspecting the plots, choose a threshold for each model and compute their validation metrics side by side. Then pick the model you want to use for your final predictions and set `MODEL_CHOICE` below.


In [ ]:
# Set your per-model thresholds after inspecting the plots above
THRESH_LR    = None  # e.g. 0.45
THRESH_LASSO = None  # e.g. 0.50

assert THRESH_LR    is not None, 'Set THRESH_LR before running this cell'
assert THRESH_LASSO is not None, 'Set THRESH_LASSO before running this cell'

def summarise(label, y_true, y_prob, threshold):
    preds = (np.array(y_prob) >= threshold).astype(int)
    print(f'{label}  (threshold={threshold})')
    print(f'  AUC:       {roc_auc_score(y_true, y_prob):.4f}')
    print(f'  Accuracy:  {accuracy_score(y_true, preds):.4f}')
    print(f'  Precision: {precision_score(y_true, preds, zero_division=0):.4f}')
    print(f'  Recall:    {recall_score(y_true, preds, zero_division=0):.4f}')
    print(f'  F1:        {f1_score(y_true, preds, zero_division=0):.4f}')

summarise('LR (selected features)', y_val, val_prob_lr,    THRESH_LR)
print()
summarise('Lasso (all features)',   y_val, val_prob_lasso, THRESH_LASSO)


In [ ]:
# Set to 'lr' or 'lasso'
MODEL_CHOICE = 'lasso'  # <-- change this to 'lr' if you prefer the 5-feature model

if MODEL_CHOICE == 'lr':
    chosen_features = selected_features
    chosen_idx = idx
elif MODEL_CHOICE == 'lasso':
    chosen_features = feature_cols  # lasso uses all features
    chosen_idx = list(range(len(feature_cols)))
else:
    raise ValueError("MODEL_CHOICE must be 'lr' or 'lasso'")

print(f'You chose: {MODEL_CHOICE}')


Refit the chosen model on the **full training set** (train + validation combined):


In [ ]:
if MODEL_CHOICE == 'lr':
    final_model = LogisticRegression(max_iter=1000, random_state=42)
    final_model.fit(X[:, chosen_idx], y)
elif MODEL_CHOICE == 'lasso':
    best_C = max(results_lasso, key=lambda r: r['val_auc'])['C']
    final_model = LogisticRegression(
        solver='saga', l1_ratio=1, C=best_C,
        max_iter=5000, random_state=42
    )
    final_model.fit(X[:, chosen_idx], y)

print('Final model fit on full training set.')


---
## 8. Make Predictions and Submit

### 8.1 Choose Your Final Threshold

You already inspected the threshold-metric trade-off in Section 7. Now set your final threshold explicitly below. **There is no default** — the cell will raise an error until you provide a value. Briefly justify your choice: what error type matters more in this context, and how does that inform the threshold you picked?


In [ ]:
# --- SET YOUR FINAL THRESHOLD HERE (do not leave as None) ---
MY_THRESHOLD = None  # e.g. 0.45

assert MY_THRESHOLD is not None, (
    'You must set MY_THRESHOLD before making predictions. '
    'Use plot_threshold_metrics() in Section 7 to inform your choice.'
)


### 8.2 Generate Test Predictions

The test set (no outcomes) is available at `/home/mlfp_shared/nij_data/test_no_outcomes.csv`. It has already been preprocessed to match the training set column schema.


In [ ]:
X_test_arr = test[feature_cols].values

test_prob = final_model.predict_proba(X_test_arr[:, chosen_idx])[:, 1]
test_pred = (test_prob >= MY_THRESHOLD).astype(int)

print(f'Test predictions: {test_pred.sum()} positive out of {len(test_pred)}')
print(f'Positive rate: {test_pred.mean():.3f}')


### 8.3 Save and Submit

Save your predictions to the shared predictions folder. Replace `YOUR_ANDREW_ID` with your actual Andrew ID:


In [ ]:
import os

ANDREW_ID = 'YOUR_ANDREW_ID'  # <-- replace with your Andrew ID

assert ANDREW_ID != "YOUR_ANDREW_ID", "Replace the placeholder with your actual andrew ID"

# DONT CHANGE ANYTHING BELOW
submission = pd.DataFrame({
    'ID':               test['ID'],
    'prob_recidivism':  test_prob,
    'pred_recidivism':  test_pred,
})

out_dir  = Path('/home/mlfp_shared/nij_data/predictions')
out_path = out_dir / f'{ANDREW_ID}.csv'

os.makedirs(out_dir, exist_ok=True)
submission.to_csv(out_path, index=False)

print(f'Submission saved to {out_path}')
print(submission.head())

That's it! Once all students have submitted, I will run the scoring script and the leaderboard will appear at `/home/mlfp_shared/nij_data/leaderboard.md`. Open it directly in VS Code — it renders as a formatted markdown table.
